In [0]:
# ============================================================
# CELL 1 — RE-READ TABLES AFTER DRIFT
# ============================================================
BORE_PORT = "8762"   # ← your bore port

jdbc_url = f"jdbc:postgresql://bore.pub:{BORE_PORT}/practice_db"
connection_props = {
    "user":     "admin",
    "password": "admin123",
    "driver":   "org.postgresql.Driver"
}

print("=" * 60)
print("📥 READING TABLES FROM POSTGRESQL")
print("=" * 60)

# Read orders
orders_df = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_props
)
print(f"\n📋 ORDERS")
print(f"   Rows:    {orders_df.count()}")
print(f"   Columns: {orders_df.columns}")
orders_df.show(5, truncate=False)

# Read customers
customers_df = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=connection_props
)
print(f"\n📋 CUSTOMERS")
print(f"   Rows:    {customers_df.count()}")
print(f"   Columns: {customers_df.columns}")
customers_df.show(5, truncate=False)

print("\n✅ Both tables read successfully!")

In [0]:
# ============================================================
# CELL 2 — DETECT DRIFT (includes detector function)
# ============================================================
import json
from pyspark.sql.functions import current_timestamp

# ── Config ──────────────────────────────────────────────────
BORE_PORT  = "8762"   # ← your bore port
DELTA_PATH = "/FileStore/cdc_practice"

jdbc_url = f"jdbc:postgresql://bore.pub:{BORE_PORT}/practice_db"
connection_props = {
    "user":     "admin",
    "password": "admin123",
    "driver":   "org.postgresql.Driver"
}

# ── Drift Detector ───────────────────────────────────────────
class SchemaDriftDetector:

    def get_saved_schema(self, table_name: str) -> dict:
        schema_path = f"{DELTA_PATH}/schemas/{table_name}"
        try:
            schema_df = (spark.read
                        .format("delta")
                        .load(schema_path)
                        .orderBy("saved_at", ascending=False)
                        .limit(1))
            row = schema_df.first()
            if row:
                return json.loads(row["schema_json"])
            return {}
        except Exception:
            return {}

    def get_source_schema(self, source_df) -> dict:
        return {
            field.name: str(field.dataType)
            for field in source_df.schema.fields
        }

    def detect_drift(self, table_name: str, source_df) -> dict:
        saved    = self.get_saved_schema(table_name)
        incoming = self.get_source_schema(source_df)

        drift = {
            "table":           table_name,
            "new_columns":     {k: v for k, v in incoming.items() if k not in saved},
            "dropped_columns": {k: v for k, v in saved.items()    if k not in incoming},
            "type_changes":    {
                k: {"from": saved[k], "to": incoming[k]}
                for k in saved
                if k in incoming and saved[k] != incoming[k]
            }
        }
        drift["has_drift"] = bool(
            drift["new_columns"]     or
            drift["dropped_columns"] or
            drift["type_changes"]
        )
        return drift

detector = SchemaDriftDetector()
print("✅ Detector ready!")

# ── Read Tables ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("📥 READING TABLES FROM POSTGRESQL")
print("=" * 60)

orders_df = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_props
)
print(f"✅ Orders:    {orders_df.count()} rows → {orders_df.columns}")

customers_df = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=connection_props
)
print(f"✅ Customers: {customers_df.count()} rows → {customers_df.columns}")




In [0]:
# ── Detect Drift ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("🔍 DETECTING SCHEMA DRIFT")
print("=" * 60)

orders_drift = detector.detect_drift("orders", orders_df)
print(f"\n📊 ORDERS DRIFT REPORT:")
print(f"-" * 40)
print(f"   Has Drift:       {orders_drift['has_drift']}")
print(f"   New Columns:     {list(orders_drift['new_columns'].keys())}")
print(f"   Dropped Columns: {list(orders_drift['dropped_columns'].keys())}")
print(f"   Type Changes:    {list(orders_drift['type_changes'].keys())}")

In [0]:

customers_drift = detector.detect_drift("customers", customers_df)
print(f"\n📊 CUSTOMERS DRIFT REPORT:")
print(f"-" * 40)
print(f"   Has Drift:       {customers_drift['has_drift']}")
print(f"   New Columns:     {list(customers_drift['new_columns'].keys())}")
print(f"   Dropped Columns: {list(customers_drift['dropped_columns'].keys())}")
print(f"   Type Changes:    {list(customers_drift['type_changes'].keys())}")

print("\n✅ Drift detection complete!")

In [0]:
# ============================================================
# CELL 3 — APPLY MAPPING FOR ORDERS (includes mapping engine)
# ============================================================
from pyspark.sql.functions import col, lit, current_timestamp, when
import json

# ── Type Map ─────────────────────────────────────────────────
TYPE_MAP = {
    "IntegerType"       : "integer",
    "LongType"          : "long",
    "StringType"        : "string",
    "DoubleType"        : "double",
    "DecimalType(10,2)" : "decimal(10,2)",
    "DecimalType(5,2)"  : "decimal(5,2)",
    "TimestampType"     : "timestamp",
    "DateType"          : "date",
    "BooleanType"       : "boolean",
    "ShortType"         : "integer",
    "FloatType"         : "double",
}

# ── Manual Mapping Engine ────────────────────────────────────
class ManualMappingEngine:

    def apply_dynamic_mapping(self, source_df, drift_report: dict):
        df            = source_df
        table_name    = drift_report["table"]
        existing_cols = [c.lower() for c in df.columns]

        print(f"\n🔧 Applying mapping for: {table_name}")
        print(f"   Existing columns: {df.columns}")

        # ── Handle NEW columns ──────────────────────────
        if drift_report["new_columns"]:
            print(f"\n   ➕ New columns:")
            for col_name, col_type in drift_report["new_columns"].items():
                spark_type = TYPE_MAP.get(col_type, "string")
                if col_name.lower() in existing_cols:
                    print(f"      + {col_name} found → casting to {spark_type}")
                    df = df.withColumn(
                        col_name,
                        when(
                            col(col_name).isNotNull(),
                            col(col_name).cast(spark_type)
                        ).otherwise(lit(None).cast(spark_type))
                    )
                else:
                    print(f"      + {col_name} missing → adding as null")
                    df = df.withColumn(
                        col_name,
                        lit(None).cast(spark_type)
                    )

        # ── Handle DROPPED columns ──────────────────────
        if drift_report["dropped_columns"]:
            print(f"\n   ➖ Dropped columns (preserving as null):")
            for col_name, col_type in drift_report["dropped_columns"].items():
                spark_type = TYPE_MAP.get(col_type, "string")
                if col_name.lower() not in existing_cols:
                    print(f"      - {col_name} → adding back as null")
                    df = df.withColumn(
                        col_name,
                        lit(None).cast(spark_type)
                    )

        # ── Handle TYPE CHANGES ─────────────────────────
        if drift_report["type_changes"]:
            print(f"\n   🔄 Type changes:")
            for col_name, change in drift_report["type_changes"].items():
                new_type = TYPE_MAP.get(change["to"], "string")
                if col_name.lower() in existing_cols:
                    print(f"      ~ {col_name}: {change['from']} → {change['to']}")
                    df = df.withColumn(
                        col_name,
                        when(
                            col(col_name).isNotNull(),
                            col(col_name).cast(new_type)
                        ).otherwise(lit(None).cast(new_type))
                    )

        # ── Audit Columns ───────────────────────────────
        version = f"{table_name}_v{hash(str(drift_report)) % 1000}"
        df = (df
            .withColumn("_schema_version", lit(version).cast("string"))
            .withColumn("_processed_at",   current_timestamp())
        )

        print(f"\n   ✅ Mapping complete!")
        print(f"   Final columns: {df.columns}")
        return df

    def generate_mapping(self, drift_report: dict):
        return json.dumps(drift_report)

    def execute_mapping(self, drift_json: str, source_df):
        drift_report = json.loads(drift_json)
        return self.apply_dynamic_mapping(source_df, drift_report)


claude_engine = ManualMappingEngine()
print("✅ Mapping Engine ready!")

# ── Apply Mapping for Orders ─────────────────────────────────
print("\n" + "=" * 60)
print("🔧 APPLYING MAPPING FOR ORDERS")
print("=" * 60)

if orders_drift["has_drift"]:
    print(f"\n⚠️  Drift detected in orders!")
    print(f"   New Columns:     {list(orders_drift['new_columns'].keys())}")
    print(f"   Dropped Columns: {list(orders_drift['dropped_columns'].keys())}")
    print(f"   Type Changes:    {list(orders_drift['type_changes'].keys())}")

    # Apply mapping
    mapping_code  = claude_engine.generate_mapping(orders_drift)
    orders_mapped = claude_engine.execute_mapping(mapping_code, orders_df)

    print(f"\n✅ Mapping applied!")
    print(f"   Columns before: {orders_df.columns}")
    print(f"   Columns after:  {orders_mapped.columns}")
    orders_mapped.show(5, truncate=False)

else:
    print("✅ No drift in orders — using original dataframe")
    orders_mapped = orders_df

In [0]:
# ============================================================
# WRITE — fixed version
# ============================================================

def write_to_delta(df, table_name: str):
    silver_table = f"silver_{table_name}"

    print(f"\n💾 Writing {table_name} → {silver_table}")
    print(f"   Rows:    {df.count()}")
    print(f"   Columns: {df.columns}")

    try:
        table_exists = spark.catalog.tableExists(silver_table)

        if table_exists:
            print(f"   🔀 Table exists → dropping and recreating")
            
            # ✅ Fix: drop first then recreate fresh
            spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
            
            (df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(silver_table))

        else:
            print(f"   🆕 Table does not exist → creating fresh")
            (df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(silver_table))

        print(f"   ✅ Successfully written to {silver_table}!")
        
        # Quick verify
        count = spark.read.table(silver_table).count()
        print(f"   ✅ Verified: {count} rows in Delta")

    except Exception as e:
        print(f"   ❌ Failed: {e}")
        raise e


# ============================================================
# RUN
# ============================================================


In [0]:
orders_df = spark.sql("select * from workspace.default.silver_orders")
customers_df = spark.sql("select * from workspace.default.silver_customers")

In [0]:
orders_df.printSchema()

In [0]:
# ============================================================
# SIMULATE DRIFT
# Step 1: Run this SQL in DBeaver first:
#
# ALTER TABLE orders ADD COLUMN discount_pct DECIMAL(5,2) DEFAULT 0.0;
# UPDATE orders SET discount_pct = 10.0 WHERE order_id = 1;
#
# Step 2: Then run this cell
# ============================================================

print("Re-reading orders & customers after schema change...")

orders_df_new = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    properties=connection_props
)
customers_df_new = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=connection_props )

print(f"New columns: {orders_df_new.columns}")
print(f"New columns:  {customers_df_new.schema}")
# You should now see discount_pct in the list!

# Write to Delta — mergeSchema handles the new column
#i want to see which columns exists in order_df_new but do not exists in orders




In [0]:
print(set(orders_df_new.columns) - set(orders_df.columns))


In [0]:
print(set(customers_df_new.columns) - set(customers_df.columns))

In [0]:
#write_to_delta(orders_df,    "orders")
#write_to_delta(customers_df, "customers")

# Verify new column is in Delta
delta_df = spark.read.table("silver_orders")
print(f"\nDelta columns now: {delta_df.columns}")
delta_df.show(5, truncate=False)

In [0]:
display(customers_df_new)

In [0]:
# Write the changes from customers_df_new onto the silver_customers table
write_to_delta(customers_df_new, "customers")

In [0]:
write_to_delta(orders_df_new, "orders")

In [0]:
# ============================================================
# CELL 6 — VERIFY FINAL DELTA TABLES
# ============================================================
print("=" * 60)
print("🔍 FINAL DELTA TABLE VERIFICATION")
print("=" * 60)

for table_name in ["orders", "customers"]:
    silver_table = f"silver_{table_name}"

    print(f"\n📋 {silver_table.upper()}")
    print("-" * 40)

    try:
        delta_df = spark.read.table(silver_table)

        print(f"   ✅ Table found in Delta!")
        print(f"   Rows:    {delta_df.count()}")
        print(f"   Columns: {delta_df.columns}")
        delta_df.show(5, truncate=False)

        # Show schema in detail
        print(f"\n   Full Schema:")
        delta_df.printSchema()

    except Exception as e:
        print(f"   ❌ Table not found: {e}")

print("\n" + "=" * 60)
print("🎉 SCHEMA EVOLUTION COMPLETE!")
print("=" * 60)